<a href="https://colab.research.google.com/github/MMQuants/MyScraper/blob/main/beautifulsoup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import csv

In [1]:
!pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 MB 52.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [playwright]


In [2]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [selenium]


In [10]:
%%shell
wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
sudo apt-get install -y ./google-chrome-stable_current_amd64.deb
pip install -q selenium pandas
google-chrome --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 54 not upgraded.
Need to get 331 kB/153 MB of archives.
After this operation, 482 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 at-spi2-common all 2.52.0-1build1 [8,674 B]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libatk1.0-0t64 amd64 2.52.0-1build1 [55.3 kB]
Get:3 

In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

url = (
    "https://www.booking.com/searchresults.html"
    "?ss=Varna%2C+Bulgaria&checkin=2026-12-15&checkout=2026-12-16"
    "&group_adults=2&no_rooms=1&group_children=0"
    "&selected_currency=EUR&lang=en-us"
)

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)
hotels_data = []


def get_text(parent, selector):
    try:
        return parent.find_element(By.CSS_SELECTOR, selector).text.strip() or "N/A"
    except Exception:
        return "N/A"


try:
    driver.get(url)
    time.sleep(8)

    # ---------- DIAGNOSTIC: what page did we actually get? ----------
    print("TITLE:", driver.title)
    print("URL:", driver.current_url)
    n_cards = len(driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="property-card"]'))
    print("CARDS:", n_cards)
    driver.save_screenshot("debug.png")
    print(driver.page_source[:1500])
    # -----------------------------------------------------------------

    if n_cards > 0:
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)

        hotels = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="property-card"]')
        print(f"Found {len(hotels)} cards")

        for hotel in hotels:
            try:
                link = hotel.find_element(By.CSS_SELECTOR, 'a[data-testid="title-link"]').get_attribute("href")
                link = link.split("?")[0]
            except Exception:
                link = "N/A"

            hotels_data.append({
                "name": get_text(hotel, '[data-testid="title"]'),
                "location": get_text(hotel, '[data-testid="address"]'),
                "distance": get_text(hotel, '[data-testid="distance"]'),
                "price": get_text(hotel, '[data-testid="price-and-discounted-price"]'),
                "rating": get_text(hotel, '[data-testid="review-score"]'),
                "url": link,
            })
    else:
        print("No property cards found. Check TITLE and debug.png to see why.")

finally:
    driver.quit()

df = pd.DataFrame(hotels_data)
display(df)


TITLE: Booking.com: Search results: Varna%2C+Bulgaria. Book your hotel now!
URL: https://www.booking.com/searchresults.html?ss=Varna%2C+Bulgaria&checkin=2026-12-15&checkout=2026-12-16&group_adults=2&no_rooms=1&group_children=0&selected_currency=EUR&lang=en-us&chal_t=1794993028418&force_referer=
CARDS: 0
<html dir="ltr" lang="en-us"><head>
    <title>Booking.com: Search results: Varna%2C+Bulgaria. Book your hotel now!</title>
    <meta data-rh="true" name="booking-verification" content="eyJyZiI6IiIsInRpIjoxNzk0OTkzMDI5LCJsYiI6Im1rdDEyM3NjLTU3NjI4MTdkLTdiMmItNGNiMC05YTYzLTVkMzNkODk1OTc4OSIsImlwIjoiMzQuMTI1LjIyLjE4OSIsImljIjoidXMiLCJ1YSI6Ik1vemlsbGEvNS4wIChXaW5kb3dzIE5UIDEwLjA7IFdpbjY0OyB4NjQpIEFwcGxlV2ViS2l0LzUzNy4zNiAoS0hUTUwsIGxpa2UgR2Vja28pIENocm9tZS8xMjAuMC4wLjAgU2FmYXJpLzUzNy4zNiIsInV0IjoidXNlciJ9">
    <meta data-rh="true" name="robots" content="noindex,follow">
    <meta data-rh="true" property="fb:pages" content="131840030178250, 1425349334428496, 117615518393985, 156584450370628

""
